# **Squad Nina da Hora - Python para Dados | Bootcamp Data Analytics 2026.1**

junho/2026

# **AJUSTES INICIAIS**

In [ ]:
# Importa as bibliotecas
import pandas as pd
import numpy as np

# Carregamento da base de dados sobre estilo de vida via Google Drive
# Armazena o id do arquivo csv e o insere na variavel da url
sheet_id = '1tH_KZnHdZ5SBjTvLEHZ7M2CMHQwl_zpc5gNkWTT9t-o'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv'

# Le o csv e armazena os dados em um data frame
df = pd.read_csv(url)
df.head(0) # Visualiza somente o cabecalho

# **QUESTÃO 1**

Mude o 'ID' para 'Identificador', corrija o nome da coluna que indica a pressão sanguínea, mude a coluna 'Ocupação' para 'Profissão', a coluna 'Categoria BMI' está em parte em inglês, substitua para 'Categoria IMC'.

In [ ]:
# Tratamento de dados - padronizacao do df
# Renomeia as colunas e visualiza somente o cabecalho
df.rename(columns={
    'ID': 'Identificador',
    'Pressão sanguíneaaaa':'Pressão sanguínea',
    'Ocupação': 'Profissão',
    'Categoria BMI': 'Categoria IMC'
}, inplace=True)

df.head(0)

# **QUESTÃO 2**

Qual é a média, a moda e a mediana de horas de sono para cada uma das profissões? ['mean', np.median, pd.Series.mode]

In [ ]:
# Perfil de sono de cada profissao
# Calcula media, mediana e moda da duracao do sono, agrupando por profissao
estatisticas_sono = df.groupby('Profissão')['Duração do sono'].agg(['mean', 'median', pd.Series.mode])

# OPCIONAL: Renomeia as colunas dos calculos
estatisticas_sono = estatisticas_sono.rename(columns={
    'mean': 'Média do Sono',
    'median': 'Mediana do Sono',
    'mode': 'Moda do Sono'
})

estatisticas_sono

# **QUESTÃO 3**

Cálculo de Impacto de Obesidade em Eng. Software

Das pessoas que atuam com engenharia de software qual a porcentagem de obesos?

In [ ]:
# Filtra apenas os dados de quem atua com eng. software
df_eng_software = df[df['Profissão'] == 'Eng. de Software'] # mascara
df_eng_software

In [ ]:
# Faz a contagem de profissionais de cada categoria de imc
cont_imc_eng = df_eng_software.groupby('Profissão')['Categoria IMC'].value_counts()

# Calcula e imprime o total de profissionais
total_imc_eng = cont_imc_eng.sum()

profissao_eng = df_eng_software['Profissão'].values[0]
print(f"Total de {profissao_eng} da Base: {total_imc_eng}")

In [ ]:
# Constroi a tabela resumo
# Transforma em df e renomeia a coluna
df_imc_eng = cont_imc_eng.reset_index(name='Total')
# Calcula as porcentagens das categorias, armazenando em uma nova coluna
df_imc_eng['Total %'] = df_imc_eng['Total'] * 100 / total_imc_eng
df_imc_eng

In [ ]:
# A partir daqui, e adicional, porque o que ja foi feito antes responde

# Funcao que busca e exibe a taxa da categoria do IMC
def retorna_imc(grupo, categoria, num_format=False):
  # Checa se a categoria nao existe nos valores de IMC
  if categoria not in grupo['Categoria IMC'].values:
    return categoria, 0 # se nao existe, o valor e zero

  # Atribui padrao para pegar o valor com porcentagem na coluna de total
  nome_coluna = 'Total %'

  if num_format: # Se formato existir (True)
    nome_coluna = 'Total' # Pega valor na coluna de total absoluto

  # Retorna categoria e seu total no padrao percetual ou absoluto
  return categoria, grupo.loc[grupo['Categoria IMC'] == categoria, nome_coluna].values[0]

# Atribui os valores retornados da funcao chamada
imc_cat_print, obesos_perc_print = retorna_imc(df_imc_eng, 'Obesidade')

# Imprime resposta
print(f'Resposta: {profissao_eng} com {imc_cat_print} corresponde a {obesos_perc_print:.1f}%')

# **QUESTÃO 4**

Comparativo de Sono: Advocacia ou Vendas vs. média geral

De acordo com os dados, advogar ou ser representante de vendas faz você dormir menos? (Use o método 'isin', considere a média)

In [ ]:
# Extrai a media e o desvio da duracao do sono de toda a base
media_sono = df['Duração do sono'].agg(['mean', 'std'])
media_sono

In [ ]:
# Filtra os dados de quem e advogado e representante de vendas
filtro_profissao = df['Profissão'].isin(['Advogado(a)', 'Representante de Vendas']) # mascara
df_adv_vendas = df[filtro_profissao]
df_adv_vendas.head()

In [ ]:
# Imprime a media geral para comparar
print(f"Média Geral de Sono da Base: {media_sono['mean']:.2f} horas, com um desvio de {media_sono['std']:.2f}")

# Extrai a media e o desvio da duracao do sono de cada profissao
media_sono_adv_vendas = df_adv_vendas.groupby('Profissão')['Duração do sono'].agg(['mean', 'std'])
media_sono_adv_vendas


In [ ]:
# A partir daqui, e adicional, porque o que ja foi feito antes responde

# Funcao que compara as medias do grupo com a media geral da base
def comparar_media_sono(medias_grupo, media_geral=media_sono['mean']):
  # Percorre cada profissao e sua media no grupo de medias
  for profissao, media_grupo in medias_grupo.items():
    # Checa se a media do grupo e menor que a media geral
    if media_grupo < media_geral:
      return profissao, media_grupo
  return

# Atribui os valores retornados da funcao chamada
profissao_print, media_print = comparar_media_sono(media_sono_adv_vendas['mean'])

# Imprime resposta
print(f"Resposta: Ser {profissao_print} faz você dormir menos (Média: {media_print:.2f}h)")


# **QUESTÃO 5**

Entre quem fez enfermagem e quem fez medicina, quem tem menos horas de sono? (Use o método 'isin', considere a média)

In [ ]:
# Cria um subconjunto apenas com as profissoes de interesse
enf_med = ['Enfermeiro(a)','Médico(a)'] # mascara
saude = df[df['Profissão'].isin(enf_med)]

# Calcula a media e o desvio da duracao do sono
media = saude.groupby('Profissão')['Duração do sono'].agg(['mean', 'std']).reset_index()

# Ordena o resultado de forma crescente
media = media.sort_values(by=('mean'))
media


In [ ]:
# Identifica o profissional com menos hora de sono
menos = media['Profissão'].iloc[0] # Seleciona o primeiro registro

# Exibe o resultado
print('O profissional com menos hora de sono é o(a)', menos)

# **QUESTÃO 6**

Faça um subconjunto com as colunas Identificador, Gênero, Idade, Pressão sanguínea e Frequência cardíaca.

In [ ]:
# Fatiamento por colunas
# Armazena colunas especificas em um novo df
subconjunto = df[
    [
        'Identificador',
        'Gênero',
        'Idade',
        'Pressão sanguínea',
        'Frequência cardíaca'
    ]
]

subconjunto.head()

# **QUESTÃO 7**

Descubra qual a profissão menos frequente no conjunto. (Use value_counts)

In [ ]:
# Calcula a ocorrencia de cada profissao (em porcentagem) - por padrao, ordena descendente
df_frequencia = df['Profissão'].value_counts(normalize=True)*100
df_frequencia

In [ ]:
# Identifica a profissao com menor ocorrencia
menos_freq = df_frequencia.index[-1] # Seleciona o ultimo registro

# Exibe o resultado
print('O profissional menos frequente na amostra de entrevistados é o(a)', menos_freq)

# **QUESTÃO 8**

Quem tem maior pressão sanguínea média, homens ou mulheres? (Considere a média)

In [ ]:
# Extrai o valor de cada pressao como inteiro, armazenando em colunas separadas
# provisoriamente
pressao = df['Pressão sanguínea'].str.split('/', expand=True).astype(int)

# Cria as colunas das pressoes no df principal com os valores extraidos
df['Sistolica'] = pressao[0]
df['Diastolica'] = pressao[1]

# Calcula a média de cada subcoluna da pressão para cada gênero
gen_pressao = df.groupby('Gênero')[['Sistolica','Diastolica']].agg(['mean', 'std']).reset_index()

# Ordena o resultado de forma crescente (media da sistolica)
gen_pressao = gen_pressao.sort_values(by=('Sistolica', 'mean'))
gen_pressao


In [ ]:
# Identifica o genero com maior media da pressa sistolica
maior = gen_pressao['Gênero'].iloc[-1] # Seleciona o ultimo registro

# Exibe o resultado
print(maior, 'é quem tem maior pressão sanguínea média na amostra de entrevistados')

# **QUESTÃO 9**

É predominante entre os participantes dormir 8 horas por dia (considere usar Moda como medida)?

In [ ]:
# Calcula a moda da base inteira para essa coluna
moda_sono = df['Duração do sono'].mode()

# Exibe o resultado na tela
# Checa se ha ou nao moda
if not moda_sono.empty:
    valor_moda = moda_sono.iloc[0] # Seleciona o ultimo registro
    print(f"A moda da duração do sono é: {valor_moda} horas.")

    # Checa se a moda e igual ao limite
    if valor_moda == 8.0:
        print("Sim! É predominante entre os participantes dormir 8 horas por dia.")
    else:
        print(f"Não! O valor mais frequente é {valor_moda} horas, portanto não é predominante dormir 8 horas.")
else:
    print("Não foi possível calcular a moda (base de dados vazia).")

# **QUESTÃO 10**

Pessoas com frequências cardíacas acima de 70 dão mais passos que pessoas com frequência cardíaca menor ou igual a 70? (Use a média)

In [ ]:
# Criação da variável para o grupo de pessoas com frequência cardiaca acima de 70
grupo_mais_70 = df['Frequência cardíaca'] > 70

# Criação da variável para o grupo de pessoas com frequência cardiaca menor ou igual a 70
grupo_70_menos = df['Frequência cardíaca'] <= 70

# Calcula a média de passos de cada variável acima
passos_mais_70 = df[grupo_mais_70]['Passos diários'].agg(['mean', 'std'])
passos_70_menos = df[grupo_70_menos]['Passos diários'].agg(['mean', 'std'])

# Exibe as médias
print(f"Média de passos para o grupo de pessoas com frequência cardíaca acima de 70 é: {passos_mais_70['mean']:.2f}")
print(f"Média de passos para o grupo de pessoas com frequência cardíaca menor ou igual a 70 é: {passos_70_menos['mean']:.2f}")
print(f"Desvio padrão do grupo das pessoas com frequência cardíaca maior que 70: {passos_mais_70['std']:.2f}")
print(f"Desvio padrão do grupo das pessoas com a frequência igual ou menor que 70: {passos_70_menos['std']:.2f}")
print('\n')
# Resposta da quetão:
if passos_mais_70['mean'] > passos_70_menos['mean']:
  print("Sim, pessoas com frequência cardíaca acima de 70, em média, dão mais passos.")
else:
  print("Não, pessoas com frequência cardíaca acima de 70, em média, não dão mais passos.")

In [ ]:
# Opcional
# Calculando a média geral de passos diários.
passos_geral = df['Passos diários'].mean()

# Comparando com as médias encontradas para anteriormente.
if passos_mais_70['mean'] > passos_geral:
    print("O grupo com frequência cardíaca acima de 70 está acima da média geral de passos.")
else:
    print("O grupo com frequência cardíaca acima de 70 está abaixo da média geral de passos.")

if passos_70_menos['mean'] > passos_geral:
    print("O grupo com frequência cardíaca menor ou igual a 70 está acima da média geral de passos.")
else:
    print("O grupo com frequência cardíaca menor ou igual a 70 está abaixo da média geral de passos.")

print('\n')
# Visualizando a diferença:
dif_mais_70 = passos_mais_70['mean'] - passos_geral
dif_70_menos = passos_70_menos['mean'] - passos_geral

print(f"Diferença do grupo > 70 para a média geral: {dif_mais_70:.2f} passos")
print(f"Diferença do grupo <= 70 para a média geral: {dif_70_menos:.2f} passos")

**Facilidades**

* O uso das bibliotecas possibilitaram a realização rápida de cálculos estatísticos
* Equilíbrio entre as tarefas
* A padronização dos exercícios e a semelhança entre os comandos utilizados facilitaram a execução das atividades
* Algumas análises, como a identificação da profissão menos frequente, puderam ser realizadas de forma direta utilizando recursos de contagem simples e ordenação dos dados
* Utilização de um ambiente compartilhado para execução das tarefas e colaboração entre os integrantes do grupo

**Dificuldades**

* Interpretação de algumas tarefas, especialmente aquelas que envolviam comparações
* Compreensão da forma correta de calcular a média da pressão arterial, considerando que os dados originais eram representados por dois valores (por exemplo, 120/80)
* A familiarização com as bibliotecas NumPy e Pandas
* Tratamento da Moda nos casos de empate
* Estabelecimento de uma conexão mais estável entre a planilha de dados e o Google Colab